In [0]:
from pyspark.sql import functions as F

GOLD_PATH = "abfss://gold@ecommercedatalakebp01.dfs.core.windows.net"

JDBC_URL = (
    "jdbc:sqlserver://ecommerce-sql-server-bp01.database.windows.net:1433;"
    "database=ecommerce-db;"
    "encrypt=true;trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=60;"
)
SQL_USER = "sqladmin"
SQL_PASSWORD = dbutils.secrets.get(scope="kv-ecommerce", key="sql-password")

In [0]:
GOLD_TABLES = ["dim_customers", "dim_products", "dim_date", "fact_orders", "fact_events"]

for table in GOLD_TABLES:
    df = spark.read.format("delta").load(f"{GOLD_PATH}/{table}")

    (
        df.write
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("dbtable", f"gold.{table}")
        .option("user", SQL_USER)
        .option("password", SQL_PASSWORD)
        .mode("overwrite")
        .save()
    )
    print(f"{table}: {df.count()} rows written")